In [ ]:
!pip install -q optuna lightgbm plotly scikit-learn pandas numpy joblib transformers torch accelerate bitsandbytes

import os
import sys
import json
import datetime
import urllib.request
from typing import Dict, List, Any, Optional
import numpy as np
import pandas as pd
import joblib
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, log_loss, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from lightgbm import LGBMClassifier
import optuna

try:
    from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
    import torch
    TRANSFORMERS_AVAILABLE = True
except ImportError:
    TRANSFORMERS_AVAILABLE = False
    print("Transformers not available.")

try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False

# **1- CONFIGURATION**

In [ ]:
RANDOM_STATE = 42
N_FOLDS = 5
TOP_N = 10
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
OUT_DIR = "output"
PLOTS_DIR = os.path.join(OUT_DIR, "plots")
LEADERBOARD_PATH = os.path.join(OUT_DIR, "local_leaderboard.md")
OOF_PATH = os.path.join(OUT_DIR, "oof_predictions.csv")
MODEL_PATH = os.path.join(OUT_DIR, "best_model.joblib")
SUB_PATH = os.path.join(OUT_DIR, "submission.csv")
AGENT_MEMORY_PATH = os.path.join(OUT_DIR, "agent_memory.json")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(PLOTS_DIR, exist_ok=True)

NUM_FEATURES = ['Age', 'SibSp', 'Parch', 'Fare', 'FamilySize', 'AgePclass', 'Fare_log1p', 'TicketCount']
CAT_FEATURES = ['Pclass', 'Sex', 'Embarked', 'Title', 'IsAlone', 'HasCabin', 'CabinDeck', 'TicketPrefix']

# **2- DATA LOADER**

In [ ]:
def download_data(url=DATA_URL, out_path="titanic.csv"):
    if not os.path.exists(out_path):
        print("Downloading dataset...")
        try:
            urllib.request.urlretrieve(url, out_path)
            print("Download complete!")
        except Exception as e:
            print(f"Download failed: {e}")
            print("Using fallback URL...")
            fallback_url = "https://raw.githubusercontent.com/agconti/kaggle-titanic/master/data/train.csv"
            urllib.request.urlretrieve(fallback_url, out_path)
    return out_path

def load_data(file_path="titanic.csv"):
    csv_path = download_data(file_path)
    df = pd.read_csv(csv_path)
    print(f"Dataset shape: {df.shape}")
    return df

# **3- FEATURE ENGINEERING**

In [ ]:
def basic_feature_engineering(df, cfg):
    df = df.copy()
    if cfg.get("use_title", True):
        df['Title'] = df['Name'].str.extract(r',\s*([^.]+)\.', expand=False).str.strip()
        rare_thresh = cfg.get("title_rare_thresh", 10)
        rare_titles = df['Title'].value_counts()[df['Title'].value_counts() < rare_thresh].index
        df['Title'] = df['Title'].replace(rare_titles, 'Rare')
    else:
        df['Title'] = 'None'

    df['FamilySize'] = df['SibSp'] + df['Parch'] + 1
    df['IsAlone'] = (df['FamilySize'] == 1).astype(int)

    if cfg.get("use_cabin_deck", True):
        df['CabinDeck'] = df['Cabin'].fillna('Missing').astype(str).str[0]
        df['CabinDeck'] = df['CabinDeck'].replace('M', 'Missing')
    else:
        df['CabinDeck'] = 'None'

    if cfg.get("use_ticket_freq", True):
        tickets = df['Ticket'].astype(str)
        ticket_prefix = tickets.str.replace(r'[0-9./]+', '', regex=True).str.strip().replace('', 'NO_PREFIX')
        df['TicketPrefix'] = ticket_prefix
        df['TicketCount'] = df['Ticket'].map(df['Ticket'].value_counts())
    else:
        df['TicketPrefix'] = 'None'
        df['TicketCount'] = 1

    df['AgePclass'] = df['Age'] * df['Pclass']
    df['Fare_log1p'] = np.log1p(df['Fare'].fillna(0))
    df['HasCabin'] = df['Cabin'].notna().astype(int)
    df['Sex_Pclass'] = df['Sex'].astype(str) + '_' + df['Pclass'].astype(str)
    df['Fare_per_Person'] = df['Fare'] / df['FamilySize']
    return df

# **4- PREPROCESSOR**

In [ ]:
def build_preprocessor(num_cols, cat_cols, cfg):
    impute_strategy = cfg.get("num_impute", "median")
    num_pipeline = Pipeline([('impute', SimpleImputer(strategy=impute_strategy))])
    if cfg.get("scale_numeric", False):
        num_pipeline.steps.append(('scale', StandardScaler()))

    try:
        cat_ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
    except TypeError:
        cat_ohe = OneHotEncoder(handle_unknown='ignore', sparse=False)

    cat_pipeline = Pipeline([
        ('impute', SimpleImputer(strategy='most_frequent')),
        ('ohe', cat_ohe)
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ('num', num_pipeline, num_cols),
            ('cat', cat_pipeline, cat_cols)
        ],
        remainder='drop'
    )
    return preprocessor

# **5- MODEL BUILDER**

In [ ]:
def make_model_builder(preprocessor, model_choice, model_params):
    def builder():
        if model_choice == 'lgb':
            clf = LGBMClassifier(**model_params)
        elif model_choice == 'rf':
            clf = RandomForestClassifier(**model_params)
        elif model_choice == 'logreg':
            clf = LogisticRegression(**model_params)
        elif model_choice == 'xgb':
            try:
                import xgboost as xgb
                clf = xgb.XGBClassifier(**model_params)
            except:
                clf = LGBMClassifier(**model_params)
        else:
            clf = LGBMClassifier(**model_params)
        pipe = Pipeline([('preprocessor', preprocessor), ('clf', clf)])
        return pipe
    return builder

# **6- OOF RUNNER**

In [ ]:
def run_oof(df, features, model_builder, cfg):
    X = df[features].copy()
    y = df['Survived'].values
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)
    oof_proba = np.zeros(len(X))
    oof_pred = np.zeros(len(X))
    fold_infos = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X, y)):
        X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
        y_tr, y_val = y[tr_idx], y[val_idx]
        model = model_builder()
        try:
            model.fit(X_tr, y_tr)
        except Exception:
            model.fit(X_tr.values, y_tr)
        try:
            proba = model.predict_proba(X_val)[:, 1]
        except Exception:
            proba = model.predict(X_val).astype(float)
        preds = (proba >= 0.5).astype(int)
        oof_proba[val_idx] = proba
        oof_pred[val_idx] = preds
        acc = accuracy_score(y_val, preds)
        ll = log_loss(y_val, proba)
        try:
            roc = roc_auc_score(y_val, proba)
        except Exception:
            roc = np.nan
        fold_infos.append({'fold': fold+1, 'acc': acc, 'logloss': ll, 'roc_auc': roc})

    overall_acc = accuracy_score(y, oof_pred)
    overall_logloss = log_loss(y, oof_proba)
    overall_roc = roc_auc_score(y, oof_proba)

    oof_df = pd.DataFrame({
        'PassengerId': df['PassengerId'],
        'Survived': df['Survived'],
        'oof_pred': oof_pred.astype(int),
        'oof_proba': oof_proba
    })
    oof_df.to_csv(OOF_PATH, index=False)
    metrics = {'acc': overall_acc, 'logloss': overall_logloss, 'roc_auc': overall_roc}
    return metrics, fold_infos

# **7- LEADERBOARD**

In [ ]:
def update_leaderboard(entry, path=LEADERBOARD_PATH, top_n=TOP_N):
    existing = []
    if os.path.exists(path):
        try:
            with open(path, 'r', encoding='utf8') as f:
                md = f.read()
            start = md.index('```json')
            end = md.index('```', start+1)
            jstr = md[start+7:end].strip()
            existing = json.loads(jstr)
        except Exception:
            existing = []

    existing.append(entry)
    existing = sorted(existing, key=lambda x: x.get('metrics', {}).get('roc_auc', 0), reverse=True)[:top_n]

    lines = ["# Local Leaderboard (OOF-based)\n\n", "Top experiments (by OOF ROC AUC):\n\n"]
    for i, e in enumerate(existing, 1):
        t = e.get('time')
        name = e.get('name')
        m = e.get('metrics', {})
        p = e.get('params', {})
        lines.append(f"## {i}. {name}  ({t})\n")
        lines.append(f"- roc_auc: {m.get('roc_auc', 0):.6f}  \n")
        lines.append(f"- acc: {m.get('acc', 0):.6f}  \n")
        lines.append(f"- logloss: {m.get('logloss', 0):.6f}\n")
        lines.append(f"- config: {json.dumps(p, sort_keys=True)}\n\n")

    lines.append("```json\n")
    lines.append(json.dumps(existing, indent=2, sort_keys=True))
    lines.append("\n```\n")

    with open(path, 'w', encoding='utf8') as f:
        f.writelines(lines)

def get_best_entry(path=LEADERBOARD_PATH):
    if not os.path.exists(path):
        return None
    try:
        with open(path, 'r', encoding='utf8') as f:
            md = f.read()
        start = md.index('```json')
        end = md.index('```', start+1)
        jstr = md[start+7:end].strip()
        entries = json.loads(jstr)
        return entries[0] if entries else None
    except Exception:
        return None

# **8- OPTIMIZER**

In [ ]:
def objective_factory(df, model_choice_fixed=None):
    def objective(trial):
        cfg = {}
        cfg['use_title'] = trial.suggest_categorical("use_title", [True, False])
        cfg['title_rare_thresh'] = trial.suggest_int("title_rare_thresh", 5, 20)
        cfg['use_cabin_deck'] = trial.suggest_categorical("use_cabin_deck", [True, False])
        cfg['use_ticket_freq'] = trial.suggest_categorical("use_ticket_freq", [True, False])
        cfg['num_impute'] = trial.suggest_categorical("num_impute", ["median", "mean"])
        cfg['scale_numeric'] = trial.suggest_categorical("scale_numeric", [False, True])

        if model_choice_fixed:
            model_choice = model_choice_fixed
        else:
            model_choice = trial.suggest_categorical("model_choice", ["lgb", "rf", "logreg"])

        if model_choice == "lgb":
            model_params = {
                "n_estimators": trial.suggest_int("lgb_n_estimators", 100, 500, step=50),
                "learning_rate": trial.suggest_float("lgb_lr", 0.01, 0.2, log=True),
                "num_leaves": trial.suggest_int("lgb_num_leaves", 7, 127),
                "random_state": RANDOM_STATE,
                "n_jobs": -1,
                "verbose": -1
            }
        elif model_choice == "rf":
            model_params = {
                "n_estimators": trial.suggest_int("rf_n_estimators", 100, 500),
                "max_depth": trial.suggest_int("rf_max_depth", 3, 30),
                "random_state": RANDOM_STATE,
                "n_jobs": -1
            }
        else:
            model_params = {
                "C": trial.suggest_float("lr_C", 1e-3, 100, log=True),
                "max_iter": 1000,
                "solver": "lbfgs",
                "random_state": RANDOM_STATE
            }

        df_fe = basic_feature_engineering(df, cfg)
        features = [f for f in (NUM_FEATURES + CAT_FEATURES) if f in df_fe.columns]
        preprocessor = build_preprocessor(
            num_cols=[c for c in features if c in NUM_FEATURES],
            cat_cols=[c for c in features if c in CAT_FEATURES],
            cfg=cfg
        )
        model_builder = make_model_builder(preprocessor, model_choice, model_params)
        metrics, fold_infos = run_oof(df_fe, features, model_builder, cfg)

        entry = {
            'time': datetime.datetime.utcnow().isoformat(),
            'name': f"trial_{trial.number}_{model_choice}",
            'metrics': metrics,
            'cfg': cfg,
            'model_choice': model_choice,
            'model_params': model_params,
            'trial_number': trial.number,
            'params': {**cfg, **model_params}
        }
        update_leaderboard(entry)
        trial.set_user_attr("metrics", metrics)
        print(f"Trial {trial.number}: {model_choice} ROC_AUC={metrics['roc_auc']:.6f} ACC={metrics['acc']:.6f}")
        return metrics['roc_auc']
    return objective

def run_optimization(df, n_trials=20, model_choice=None):
    study = optuna.create_study(
        direction="maximize",
        study_name="titanic_autonomous",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
    )
    obj = objective_factory(df, model_choice_fixed=model_choice)
    study.optimize(obj, n_trials=n_trials, show_progress_bar=True)
    return study

# **9- TRAINER**

In [ ]:
def train_best_on_full(df, best_entry):
    cfg = best_entry.get('cfg', {})
    model_choice = best_entry.get('model_choice', 'lgb')
    model_params = best_entry.get('model_params', {})

    df_fe = basic_feature_engineering(df, cfg)
    features = [f for f in (NUM_FEATURES + CAT_FEATURES) if f in df_fe.columns]
    preprocessor = build_preprocessor(
        num_cols=[c for c in features if c in NUM_FEATURES],
        cat_cols=[c for c in features if c in CAT_FEATURES],
        cfg=cfg
    )
    model_builder = make_model_builder(preprocessor, model_choice, model_params)
    model = model_builder()

    X_full = df_fe[features]
    y_full = df_fe['Survived'].values
    try:
        model.fit(X_full, y_full)
    except Exception:
        model.fit(X_full.values, y_full)

    joblib.dump({'model': model, 'features': features, 'cfg': cfg}, MODEL_PATH)

    try:
        proba = model.predict_proba(X_full)[:, 1]
    except Exception:
        proba = model.predict(X_full).astype(float)
    preds = (proba >= 0.5).astype(int)
    sub = pd.DataFrame({'PassengerId': df_fe['PassengerId'], 'Survived': preds})
    sub.to_csv(SUB_PATH, index=False)
    return MODEL_PATH

# **10- ENHANCED OPEN-SOURCE LLM**

In [ ]:
class EnhancedOpenSourceLLM:
    def __init__(self, model_name="microsoft/DialoGPT-medium"):
        if not TRANSFORMERS_AVAILABLE:
            raise ImportError("Transformers not available.")

        print(f"Loading LLM: {model_name}...")
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map="auto" if torch.cuda.is_available() else None
        )
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        if not torch.cuda.is_available():
            self.model.to(self.device)
        self.model.eval()

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def predict(self, prompt: str, max_new_tokens: int = 150) -> str:
        try:
            inputs = self.tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
            inputs = {k: v.to(self.device) for k, v in inputs.items()}

            with torch.no_grad():
                outputs = self.model.generate(
                    **inputs,
                    max_new_tokens=max_new_tokens,
                    do_sample=True,
                    temperature=0.5,
                    top_p=0.9,
                    pad_token_id=self.tokenizer.eos_token_id
                )

            response = self.tokenizer.decode(outputs[0], skip_special_tokens=True)
            response = response[len(prompt):].strip()

            if not response or len(response) < 3:
                return self._fallback_response(prompt)

            return response[:100]

        except Exception as e:
            print(f"⚠️ LLM prediction failed: {e}")
            return self._fallback_response(prompt)

    def _fallback_response(self, prompt: str) -> str:
        prompt_lower = prompt.lower()
        if "hyperparameter" in prompt_lower or "tune" in prompt_lower:
            return "hyperparameter_tuning"
        elif "feature" in prompt_lower or "engineer" in prompt_lower:
            return "feature_engineering"
        elif "model" in prompt_lower or "train" in prompt_lower:
            return "train_model"
        else:
            return "evaluate"

# **11- IMPROVED AGENT**

In [ ]:
class ImprovedMLAgent:
    def __init__(self, use_llm=True, model_name="microsoft/DialoGPT-medium"):
        self.use_llm = use_llm and TRANSFORMERS_AVAILABLE
        self.llm = None

        if self.use_llm:
            try:
                self.llm = EnhancedOpenSourceLLM(model_name)
                print(f"✅ LLM loaded successfully!")
            except Exception as e:
                print(f"⚠️ Could not load LLM: {e}")
                self.use_llm = False
                print("Using rule-based reasoning fallback")

        self.memory = []
        self.experiment_history = []
        self.best_roc = 0.0
        self.best_model = None
        self.current_df = None
        self.no_improvement_counter = 0
        self.actions_taken = set()

    def analyze_dataset(self, df):
        return {
            "shape": df.shape,
            "columns": list(df.columns),
            "missing_count": df.isnull().sum().sum(),
            "target": df.columns[-1] if df.shape[1] > 1 else "unknown"
        }

    def get_state(self):
        return {
            "best_roc": self.best_roc,
            "best_model": self.best_model,
            "actions_taken": list(self.actions_taken),
            "no_improvement_counter": self.no_improvement_counter,
            "experiments_count": len(self.experiment_history)
        }

    def decide_next_action(self, df, analysis):
        state = self.get_state()

        if self.no_improvement_counter >= 3 and len(self.experiment_history) > 3:
            return {"action": "stop", "reason": "No improvement in last 3 steps"}

        context = f"""
Dataset shape: {analysis['shape']}
Missing values: {analysis['missing_count']}
Best ROC: {state['best_roc']:.4f}
Best Model: {state['best_model']}
Actions taken: {state['actions_taken']}
Steps without improvement: {state['no_improvement_counter']}

Choose next action from: feature_engineering, hyperparameter_tuning, train_model, evaluate
Return only one action name.
"""

        if self.use_llm and self.llm:
            try:
                response = self.llm.predict(context, max_new_tokens=30)
                response_lower = response.lower().strip()
                for action in ["feature_engineering", "hyperparameter_tuning", "train_model", "evaluate", "stop"]:
                    if action in response_lower:
                        return {"action": action, "reason": "LLM recommendation"}
            except Exception as e:
                print(f"⚠️ LLM decision failed: {e}")

        return self._fallback_decision(state, analysis)

    def _fallback_decision(self, state, analysis):
        if not self.actions_taken or "feature_engineering" not in self.actions_taken:
            if analysis['missing_count'] > 0 or len(self.actions_taken) == 0:
                return {"action": "feature_engineering", "reason": "Initial feature engineering"}

        if "hyperparameter_tuning" not in self.actions_taken and state['best_roc'] < 0.85:
            return {"action": "hyperparameter_tuning", "reason": "Need optimization"}

        if "train_model" not in self.actions_taken:
            return {"action": "train_model", "reason": "Train initial model"}

        if state['best_roc'] < 0.88 and "hyperparameter_tuning" not in self.actions_taken:
            return {"action": "hyperparameter_tuning", "reason": "Room for improvement"}

        if state['no_improvement_counter'] >= 2:
            return {"action": "train_model", "reason": "Try different model"}

        return {"action": "evaluate", "reason": "Check current performance"}

    def execute_feature_engineering(self, df):
        print("🔧 Executing Feature Engineering...")
        cfg = {
            "use_title": True,
            "title_rare_thresh": 10,
            "use_cabin_deck": True,
            "use_ticket_freq": True,
            "num_impute": "median",
            "scale_numeric": False
        }
        df_fe = basic_feature_engineering(df, cfg)
        print(f"✅ Feature engineering done. Columns: {df_fe.shape[1]}")
        return df_fe

    def execute_hyperparameter_tuning(self, df, n_trials=10):
        print("🔧 Executing Hyperparameter Tuning...")
        study = run_optimization(df, n_trials=n_trials)
        best_trial = study.best_trial
        if best_trial.value > self.best_roc:
            self.best_roc = best_trial.value
            self.best_model = best_trial.params.get('model_choice', 'lgb')
            self.no_improvement_counter = 0
        else:
            self.no_improvement_counter += 1
        print(f"✅ Best ROC AUC: {best_trial.value:.6f}")
        return df

    def execute_train_model(self, df, model_choice=None):
        print("🤖 Training Model...")

        available_models = ['lgb', 'rf', 'logreg']
        if model_choice is None:
            tried_models = [self.best_model] if self.best_model else []
            for m in available_models:
                if m not in tried_models:
                    model_choice = m
                    break
            if model_choice is None:
                model_choice = 'lgb'

        cfg = {"use_title": True, "title_rare_thresh": 10,
               "use_cabin_deck": True, "use_ticket_freq": True,
               "num_impute": "median", "scale_numeric": False}
        df_fe = basic_feature_engineering(df, cfg)
        features = [f for f in (NUM_FEATURES + CAT_FEATURES) if f in df_fe.columns]
        preprocessor = build_preprocessor(
            num_cols=[c for c in features if c in NUM_FEATURES],
            cat_cols=[c for c in features if c in CAT_FEATURES],
            cfg=cfg
        )

        if model_choice == 'lgb':
            model_params = {"n_estimators": 300, "learning_rate": 0.05,
                           "num_leaves": 31, "random_state": RANDOM_STATE}
        elif model_choice == 'rf':
            model_params = {"n_estimators": 300, "max_depth": 15,
                           "random_state": RANDOM_STATE}
        else:
            model_params = {"C": 1.0, "max_iter": 1000,
                           "random_state": RANDOM_STATE}

        model_builder = make_model_builder(preprocessor, model_choice, model_params)
        metrics, _ = run_oof(df_fe, features, model_builder, cfg)

        if metrics['roc_auc'] > self.best_roc:
            self.best_roc = metrics['roc_auc']
            self.best_model = model_choice
            self.no_improvement_counter = 0
        else:
            self.no_improvement_counter += 1

        print(f"✅ {model_choice} ROC AUC: {metrics['roc_auc']:.6f}")
        return df

    def execute_evaluate(self, df):
        print("📊 Evaluating...")
        best_entry = get_best_entry()
        if best_entry:
            roc = best_entry['metrics']['roc_auc']
            model = best_entry['model_choice']
            if roc > self.best_roc:
                self.best_roc = roc
                self.best_model = model
                self.no_improvement_counter = 0
            print(f"📊 Best model: {model} with ROC AUC: {roc:.6f}")
        else:
            print("📊 No model found yet.")
        return df

    def execute_stop(self, df):
        print("🛑 Stopping agent...")
        return df

    def run(self, df, max_steps=10):
        print("🤖 Starting Improved ML Agent")
        print("="*60)

        self.current_df = df

        for step in range(max_steps):
            print(f"\n📋 Step {step+1}/{max_steps}")

            analysis = self.analyze_dataset(self.current_df)
            decision = self.decide_next_action(self.current_df, analysis)

            action = decision['action']
            reason = decision.get('reason', 'No reason provided')
            print(f"🎯 Action: {action} (Reason: {reason})")

            if action == 'feature_engineering':
                self.current_df = self.execute_feature_engineering(self.current_df)
                self.actions_taken.add('feature_engineering')

            elif action == 'hyperparameter_tuning':
                self.current_df = self.execute_hyperparameter_tuning(self.current_df, n_trials=8)
                self.actions_taken.add('hyperparameter_tuning')

            elif action == 'train_model':
                self.current_df = self.execute_train_model(self.current_df)
                self.actions_taken.add('train_model')

            elif action == 'evaluate':
                self.current_df = self.execute_evaluate(self.current_df)
                self.actions_taken.add('evaluate')

            elif action == 'stop':
                print("🛑 Stopping early based on decision")
                break

            self.experiment_history.append({
                "step": step + 1,
                "action": action,
                "best_roc": self.best_roc,
                "best_model": self.best_model,
                "timestamp": datetime.datetime.now().isoformat()
            })

            with open(AGENT_MEMORY_PATH, 'w') as f:
                json.dump({
                    "history": self.experiment_history,
                    "best_roc": self.best_roc,
                    "best_model": self.best_model
                }, f, indent=2)

        print("\n" + "="*60)
        print("✅ Agent Execution Complete!")
        print("="*60)
        print(f"🏆 Best ROC AUC: {self.best_roc:.6f}")
        print(f"🏆 Best Model: {self.best_model}")
        print(f"📊 Total Steps: {len(self.experiment_history)}")

        return {
            "best_roc": self.best_roc,
            "best_model": self.best_model,
            "history": self.experiment_history
        }

# **12- MAIN EXECUTION**

In [ ]:
def run_agent_mode_enhanced():
    print("="*60)
    print("RUNNING ENHANCED AGENT MODE")
    print("="*60)

    df = load_data()

    try:
        agent = ImprovedMLAgent(use_llm=True, model_name="microsoft/DialoGPT-medium")
    except Exception as e:
        print(f"⚠️ Could not load LLM: {e}")
        print("Using rule-based agent")
        agent = ImprovedMLAgent(use_llm=False)

    results = agent.run(df, max_steps=10)

    if results['best_roc'] > 0.85:
        print("\n🚀 Training final best model...")
        best_entry = get_best_entry()
        if best_entry:
            train_best_on_full(df, best_entry)
            print(f"✅ Saved best model to {MODEL_PATH}")
            print(f"✅ Saved submission to {SUB_PATH}")

    print(f"\n✅ All done. Check {OUT_DIR}/ directory")
    print(f"✅ Leaderboard: {LEADERBOARD_PATH}")


def run_automated_mode_enhanced(n_trials=20):
    print("="*60)
    print("RUNNING AUTOMATED MODE")
    print("="*60)

    df = load_data()
    study = run_optimization(df, n_trials=n_trials)

    best_entry = get_best_entry()
    if best_entry:
        print(f"\n🏆 Best entry: {best_entry['name']} with ROC_AUC={best_entry['metrics']['roc_auc']:.6f}")
        print("Training best model on full dataset...")
        train_best_on_full(df, best_entry)
        print(f"✅ Saved best model to {MODEL_PATH}")
        print(f"✅ Saved submission to {SUB_PATH}")

    print(f"\n✅ Automated mode complete. Check {OUT_DIR}/ directory")

# **EXECUTION**

In [ ]:
if __name__ == "__main__":
    print("\n" + "="*60)
    print("AGENTIC-ML-OPTIMIZER v2.1")
    print("="*60)
    print("\nChoose mode:")
    print("1. Enhanced Agent Mode (with DialoGPT LLM)")
    print("2. Automated Mode (Optuna Optimization)")
    print("3. Quick Test (5 trials)")
    print("")

    try:
        choice = input("Enter choice (1/2/3) [default=1]: ").strip() or "1"
    except:
        choice = "1"

    if choice == "1":
        print("\n🚀 Starting Enhanced Agent Mode...")
        run_agent_mode_enhanced()
    elif choice == "2":
        print("\n🚀 Starting Automated Mode...")
        run_automated_mode_enhanced(n_trials=20)
    elif choice == "3":
        print("\n🚀 Quick Test Mode...")
        run_automated_mode_enhanced(n_trials=5)
    else:
        print("Invalid choice. Running Enhanced Agent Mode...")
        run_agent_mode_enhanced()

    print("\n" + "="*60)
    print("✅ Done! Check output/ directory for results")
    print("="*60)